In [ ]:
! git clone https://github.com/nhsbsa-data-analytics/coffee-and-coding


# Introduction

In this coffee and coding session, we build an intent-based chatbot system using Natural Language Processing (NLP) and Machine Learning.

The goal is to demonstrate how conversational systems understand user queries and return meaningful responses in a structured way.

Unlike generative AI systems, this chatbot does not create responses dynamically. Instead, it:

Classifies user input into predefined intents (such as services, claims, benefits, and eligibility)
Maps each intent to a predefined response
Ensures consistent, explainable, and reliable outputs

We then extend this system using Named Entity Recognition (NER) to extract key details such as dates, amounts, and document types from user input.

Finally, we demonstrate how the same NLP techniques can be applied to a real-world scenario—extracting structured information from payslips using OCR + NLP.

This notebook provides a complete, end-to-end view of how NLP powers real-world systems in domains such as healthcare, finance, and public services.

# Text Preprocessing in NLP

Text preprocessing is a crucial step in Natural Language Processing (NLP). It prepares raw, unstructured text so that machine learning models can understand and process it effectively.

In real-world applications like chatbots, search engines, and support systems, user input is often messy. It may include spelling mistakes, slang, abbreviations, punctuation, or emojis. Preprocessing helps clean and standardize this input.

This step ensures that the text is transformed into a consistent format, allowing models to learn patterns more effectively and make accurate predictions.

---

## Importance of Text Preprocessing

### 1. Improves Data Quality  
Preprocessing removes noise such as punctuation, stray characters, emojis, and unnecessary words, resulting in cleaner input for modelling.

### 2. Boosts Model Performance  
Cleaner text allows feature extraction methods such as TF-IDF to produce more meaningful representations, which in turn improves classification accuracy.

### 3. Reduces Complexity  
Simplifying text lowers the computational cost and helps algorithms run efficiently.

---

# NLP Pipeline Used in This Notebook

Below is the complete preprocessing pipeline implemented in this project to convert raw text into machine‑readable features.

### 1. Raw Input  
Example user query:  
"Hi, can u tell me more about NHSBSA services??"

### 2. Text Cleaning  
Operations include:  
- Converting to lowercase  
- Removing punctuation and special characters  
- Removing emojis and unnecessary symbols  

**Output example:**  
`tell me more about nhsbsa services and how claims work`

### 3. Tokenization  
Splitting cleaned text into individual tokens.  
**Example:**  
`["tell", "me", "more", "about", "nhsbsa", "services"]`

### 4. Stopword Removal  
Removing high‑frequency words that contribute little meaning.  
**Example:**  
`["tell", "nhsbsa", "services"]`

### 5. Feature Extraction (TF‑IDF)  
Text is transformed into numerical features using Term Frequency–Inverse Document Frequency, allowing machine learning algorithms to process it.

### 6. Intent/Label Classification  
“Intent classification predicts the user’s purpose (‘claims’, ‘services’, etc.) while entity extraction identifies specific values like dates, amounts, or document types.”


A machine learning classifier predicts the intent category of the query.  
Possible intents include:  
- services  
- claims  
- benefits  
- eligibility  

**Example Prediction:**  
Input: "how claims work"  
Output: `claims`
Here, the **intent represents the intent** of the user.

### 7. Response Generation  
Based on the predicted intent, a predefined response is returned, simulating a basic intent-based chatbot.

---

# Key Idea

intent-based (intent-based) chatbot using classification + predefined responses

This workflow demonstrates how a classical intent-based chatbot operates:

**Input → Preprocessing → Feature Extraction → intent Classification → Response**

It is important to note that this is not a generative model like ChatGPT.  
Instead, it is a rule‑based + classification‑based system that uses predefined intents and responses.

This makes it an excellent foundation for understanding the essential building blocks behind conversational AI systems.


## 0) Environment Setup

We import only core libraries (NLTK + scikit-learn). Optional extras are auto‑disabled if not installed.
This keeps the demo reliable on most machines.


In [ ]:
import spacy
from spacy.training.example import Example
import spacy.cli # Import spacy.cli for explicit downloads
spacy.cli.download("en_core_web_sm")
# Removed: spacy.cli.download("en_lookups_data") - Requires explicit pip install in setup cell for compatibility
!pip install pyspellchecker
# Core imports

import re
import string
from typing import List

# NLP basics
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Optional lemmatization
try:
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)
    from nltk.stem import WordNetLemmatizer
    HAS_LEMMA = True
except Exception:
    HAS_LEMMA = False

# Optional extras (gracefully disabled if unavailable)
try:
    import emoji
    HAS_EMOJI = True
except Exception:
    HAS_EMOJI = False

try:
    import contractions
    HAS_CONTRACTIONS = True
except Exception:
    HAS_CONTRACTIONS = False

try:
    from spellchecker import SpellChecker
    HAS_SPELL = True
except Exception:
    HAS_SPELL = False

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

import numpy as np
import matplotlib.pyplot as plt


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 1) Mini Query Dataset

A small, intentionally messy dataset to showcase cleaning: slang, typos, emojis, contractions, punctuation, and domain terms.
We also attach **weak intents** (simple heuristics) to keep this demo fully self‑contained.

**intents**: `services`, `claims`, `benefits`, `eligibility`


In [ ]:
corpus = [
"Can you tell me which NHSBSA service processes GP payments for the April 2026 period?",
"Which NHSBSA team handles PPC renewals for prescriptions costing over £100?",
"Is the NHSBSA dental services helpline open on 15 May 2026?",
"Where do I find information about maternity exemption services offered by NHSBSA?",
"How do I claim a refund for the £42 dental treatment I paid for on 12 Feb 2026?",
"Where should I upload receipts to claim back £25 for travel on 03 March 2026?",
"Can I submit my optical voucher claim for £80 through the NHSBSA portal?",
"Is there a deadline to claim a reimbursement for treatment received in January 2026?",
"What benefits are available for students earning below £1,200 per month?",
"Do NHS optical vouchers cover £50 frames or only £25 ones?",
"Are there any benefits for patients receiving treatment between 01 Jan 2026 and 31 Jan 2026?",
"Does the Learning Support Fund cover placement travel above £30 weekly?",
"Am I eligible for an HC2 certificate if my income in 2025–2026 was below £6,000?",
"Does student status for the 2026 academic year qualify me for help with health costs?",
"Is a part‑time worker earning £900 monthly eligible for the NHS Low Income Scheme?",
"Do pensioners aged over 65 automatically qualify for free prescriptions in 2026?"
]

# Simple heuristic intenting for demo

def weak_intent(text: str) -> str:
    t = text.lower()
    if any(k in t for k in ["eligib", "hc2", "criteria", "student status", "maternity exemption"]):
        return "eligibility"
    if any(k in t for k in ["claim", "reimbursement", "refund", "submit"]):
        return "claims"
    if any(k in t for k in ["benefit", "voucher", "support"]):
        return "benefits"
    return "services"

intents = [weak_intent(x) for x in corpus]

for i, (text, lab) in enumerate(list(zip(corpus, intents))[:5], start=1):
    print(f"{i:02d}. {text}  ->  {lab}")

01. Can you tell me which NHSBSA service processes GP payments for the April 2026 period?  ->  services
02. Which NHSBSA team handles PPC renewals for prescriptions costing over £100?  ->  services
03. Is the NHSBSA dental services helpline open on 15 May 2026?  ->  services
04. Where do I find information about maternity exemption services offered by NHSBSA?  ->  eligibility
05. How do I claim a refund for the £42 dental treatment I paid for on 12 Feb 2026?  ->  claims


## 2) Cleaning Pipeline — Detailed Steps

We’ll transform raw user text into a normalized, model‑friendly form. Each step is small, but together they reduce noise and improve signal.

1. **Lowercasing** – standardizes text ("NHSBSA" → "nhsbsa") while keeping semantics.
2. **HTML/URL removal** – strips tags and links that add noise (regex‑based to avoid extra deps).
3. **Emoji demojize (optional)** – converts emojis to words (e.g., `😅` → `:grinning_face_with_sweat:`) so models can learn their sentiment/intent hints.
5. **Character filtering** – keep alphanumerics so domain tokens like **hc2** survive; remove stray symbols.
6. **Tokenization** – split into word tokens (`nltk.word_tokenize`).
7. **Stopword removal** – drop very common words (e.g., "the", "is") that rarely add intent.
8. **Lemmatization** – map inflected forms to base lemmas (e.g., "claims" → "claim").
9. **Spell correction** – fix misspellings, but use sparingly (domain terms may be over‑corrected).

> The functions below implement these steps and are composed in `clean_pipeline(...)`.


In [ ]:
nltk.download('punkt_tab', quiet=True) # Explicitly download punkt_tab
STOP_WORDS = set(stopwords.words('english'))

# Minimal HTML remover (regex) to keep dependencies tiny
HTML_TAG_RE = re.compile(r"<.*?>")
URL_RE = re.compile(r"http\S+|www\.\S+")
NON_ALNUM_SPACE_RE = re.compile(r"[^a-z0-9\s]")  # keep a-z, 0-9, and space
MULTISPACE_RE = re.compile(r"\s+")


def strip_html(text: str) -> str:
    return HTML_TAG_RE.sub(" ", text)


def basic_clean(text: str) -> str:
    """Lowercase, strip HTML/URLs, keep alphanumerics, collapse spaces."""
    text = text.lower()
    text = strip_html(text)
    text = URL_RE.sub(" ", text)
    text = NON_ALNUM_SPACE_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text



def demojize(text: str) -> str:
    if HAS_EMOJI:
        return emoji.demojize(text, language='en')
    return text


def tokenize(text: str) -> List[str]:
    return word_tokenize(text)


def remove_stopwords(tokens: List[str]) -> List[str]:
    return [t for t in tokens if t not in STOP_WORDS]


def lemmatize_tokens(tokens: List[str]) -> List[str]:
    if not HAS_LEMMA:
        return tokens
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(t) for t in tokens]


def spell_correct_tokens(tokens: List[str]) -> List[str]:
    if not HAS_SPELL:
        return tokens
    sp = SpellChecker()
    return [sp.correction(t) if t.isalpha() and t not in ["nhsbsa","hc2"] else t for t in tokens]


def clean_pipeline(text: str,HAS_EMOJI: bool = True, do_spell: bool = False, do_lemma: bool = True) -> str:
    if HAS_EMOJI:
        text = demojize(text)
    text = basic_clean(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    if do_spell:
        tokens = spell_correct_tokens(tokens)
    if do_lemma:
        tokens = lemmatize_tokens(tokens)
    return " ".join(tokens)


RAW:      Can you tell me which NHSBSA service processes GP payments for the April 2026 period?
CLEANED:  tell nhsbsa service process gp payment april 2026 period


## Single Prompt Walkthrough (What the Model Sees)
This section demonstrates how raw user input is progressively cleaned and transformed, allowing us to observe exactly what the model receives for processing.

In [ ]:
example = "Hi, can u tell me more about NHSBSA services?? I kinda want to know how claims work 😅"
print("Step 1: Raw\n", example)


ex2 = demojize(example)
ex2 = basic_clean(ex2)
print("\nStep 2: Cleaned\n", ex2)

tokens = word_tokenize(ex2)
print("\nStep 3: Tokens\n", tokens)

no_stop = remove_stopwords(tokens)
print("\nStep 4: Stopwords removed\n", no_stop)

lem = lemmatize_tokens(no_stop)
print("\nStep 5: Lemmas (if available)\n", lem)


Step 1: Raw
 Hi, can u tell me more about NHSBSA services?? I kinda want to know how claims work 😅

Step 2: Cleaned
 hi can u tell me more about nhsbsa services i kinda want to know how claims work

Step 3: Tokens
 ['hi', 'can', 'u', 'tell', 'me', 'more', 'about', 'nhsbsa', 'services', 'i', 'kinda', 'want', 'to', 'know', 'how', 'claims', 'work']

Step 4: Stopwords removed
 ['hi', 'u', 'tell', 'nhsbsa', 'services', 'kinda', 'want', 'know', 'claims', 'work']

Step 5: Lemmas (if available)
 ['hi', 'u', 'tell', 'nhsbsa', 'service', 'kinda', 'want', 'know', 'claim', 'work']


## 4) Clean the Entire Corpus

We now apply the same pipeline to all prompts. Notice how different spellings/slang map to consistent tokens while keeping domain terms like **hc2**.


In [ ]:
cleaned_corpus = [clean_pipeline(doc, do_spell=False) for doc in corpus]
for raw, cl in list(zip(corpus, cleaned_corpus))[:8]:
    print(f"- RAW: {raw}\n  CLEAN: {cl}\n")


- RAW: Can you tell me which NHSBSA service processes GP payments for the April 2026 period?
  CLEAN: tell nhsbsa service process gp payment april 2026 period

- RAW: Which NHSBSA team handles PPC renewals for prescriptions costing over £100?
  CLEAN: nhsbsa team handle ppc renewal prescription costing 100

- RAW: Is the NHSBSA dental services helpline open on 15 May 2026?
  CLEAN: nhsbsa dental service helpline open 15 may 2026

- RAW: Where do I find information about maternity exemption services offered by NHSBSA?
  CLEAN: find information maternity exemption service offered nhsbsa

- RAW: How do I claim a refund for the £42 dental treatment I paid for on 12 Feb 2026?
  CLEAN: claim refund 42 dental treatment paid 12 feb 2026

- RAW: Where should I upload receipts to claim back £25 for travel on 03 March 2026?
  CLEAN: upload receipt claim back 25 travel 03 march 2026

- RAW: Can I submit my optical voucher claim for £80 through the NHSBSA portal?
  CLEAN: submit optical voucher cla

## 5) Mini intent Classifier (TF‑IDF + LinearSVC)

We split the data, build a Pipeline (TF‑IDF → classifier), train, and view a quick report.
`LinearSVC` tends to perform well on small, sparse text problems.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    cleaned_corpus, intents, test_size=0.3, random_state=42, stratify=intents
)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=1, max_df=0.95)),
    ("clf", LinearSVC(class_weight="balanced"))
])

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

print("=== Classification Report ===")
print(classification_report(y_test, pred, digits=3))


=== Classification Report ===
              precision    recall  f1-score   support

    benefits      0.000     0.000     0.000         1
      claims      0.000     0.000     0.000         1
 eligibility      0.000     0.000     0.000         2
    services      1.000     1.000     1.000         1

    accuracy                          0.200         5
   macro avg      0.250     0.250     0.250         5
weighted avg      0.200     0.200     0.200         5



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 6) Live Demo: Predict & Respond

Try a few prompts. For a true live moment, edit the `tests` list below and re‑run. The cleaning pipeline runs at inference too.


In [ ]:
intent_RESPONSES = {
    "services":   "NHSBSA provides services like payments administration, PPCs, dental and optical support.",
    "claims":     "You can submit NHSBSA claims via the online portal; timelines vary by claim type and evidence.",
    "benefits":   "Depending on your circumstances, you may qualify for support such as optical vouchers or PPC discounts.",
    "eligibility":"Eligibility may depend on income, age, exemptions (e.g., HC2), student status, or specific medical criteria."
}

def predict_intent(user_text: str):
    cleaned = clean_pipeline(user_text)
    intent = pipe.predict([cleaned])[0]
    return intent, cleaned

def generate_response(intent: str) -> str:
    return intent_RESPONSES.get(intent, "Could you share a bit more detail so I can help?")

# Try a few

tests = [

    "Can you give me an overview of services provided by NHSBSA?",
    "What services are available for people who need help with health-related payments?",


    "How do I claim a refund for a dental treatment I paid for?",
    "Where should I upload my receipts to claim back the cost of my glasses?",


    "What support is available to help with optical costs?",
    "Are there any benefits available for people who pay for prescriptions regularly?",


    "Am I eligible for help with health costs as a full-time student?",
    "What are the eligibility criteria for receiving an HC2 certificate?"
]

for t in tests:
    intent, cleaned = predict_intent(t)
    print(f"\nUser: {t}\nCleaned: {cleaned}\nPredicted intent: {intent}\nResponse: {generate_response(intent)}")



User: Can you give me an overview of services provided by NHSBSA?
Cleaned: give overview service provided nhsbsa
Predicted intent: services
Response: NHSBSA provides services like payments administration, PPCs, dental and optical support.

User: What services are available for people who need help with health-related payments?
Cleaned: service available people need help health related payment
Predicted intent: services
Response: NHSBSA provides services like payments administration, PPCs, dental and optical support.

User: How do I claim a refund for a dental treatment I paid for?
Cleaned: claim refund dental treatment paid
Predicted intent: claims
Response: You can submit NHSBSA claims via the online portal; timelines vary by claim type and evidence.

User: Where should I upload my receipts to claim back the cost of my glasses?
Cleaned: upload receipt claim back cost glass
Predicted intent: claims
Response: You can submit NHSBSA claims via the online portal; timelines vary by claim t

##Named Entity Recognition (NER) — Extending the NLP Pipeline (Based on Our Training Queries)
Our model has focused on **intent** **classification** :

✔“*What is the user trying to do*?”

But the training examples in our dataset also contain entities—important pieces of information that users mention naturally when asking a question.
NER (Named Entity Recognition) answers a different question:

✔ “*What specific information does the user mention?*”

This is essential in real workflows like NHSBSA claims, benefits, payments, exemptions, and document processing.

In the context of above queries , NER helps extract:

Dates (25 March 2026)

Money / Amounts ("£40",Money )

Organisations ("NHSBSA",ORG)

Document Types / Certificates ("HC2", CERTIFICATE)

Numeric Identifiers ("QQ123456A", NI NUMBER)


#Why NER Matters in This Demo

Chatbot currently answers:

“What is the user asking about?”

But NER answers:

“What specific pieces of information did the user mention?”

Inorder to get more accurate answers for the queries NER will be used.

#Example:

**Input**:

"How do I claim a refund for a dental treatment I paid £40 for on 12 Feb 2026?"

Intent: claims
Entities extracted: £40 → MONEY
12 Feb 2026 → DATE
dental treatment → SERVICE


#NER Training Data (Annotated for Corpus)

Entities that we want from the corpus to improve the response:

ORG

MONEY

DATE

PRODUCT

In [ ]:

TRAIN_DATA = [

("Can you tell me which NHSBSA service processes GP payments for the April 2026 period?",
 {"entities": [("NHSBSA", "ORG"), ("GP", "ORG"), ("April 2026", "DATE")]}),

("Which NHSBSA team handles PPC renewals for prescriptions costing over £100?",
 {"entities": [("NHSBSA", "ORG"), ("£100", "MONEY")]}),

("Is the NHSBSA dental services helpline open on 15 May 2026?",
 {"entities": [("NHSBSA", "ORG"), ("15 May 2026", "DATE")]}),

("Where do I find information about maternity exemption services offered by NHSBSA?",
 {"entities": [("maternity exemption", "PRODUCT"), ("NHSBSA", "ORG")]}),

("How do I claim a refund for the £42 dental treatment I paid for on 12 Feb 2026?",
 {"entities": [("£42", "MONEY"), ("12 Feb 2026", "DATE")]}),

("Where should I upload receipts to claim back £25 for travel on 03 March 2026?",
 {"entities": [("£25", "MONEY"), ("03 March 2026", "DATE")]}),

("Can I submit my optical voucher claim for £80 through the NHSBSA portal?",
 {"entities": [("£80", "MONEY"), ("NHSBSA", "ORG"), ("optical voucher", "PRODUCT")]}),

("Is there a deadline to claim a reimbursement for treatment received in January 2026?",
 {"entities": [("January 2026", "DATE")]}),

("What benefits are available for students earning below £1,200 per month?",
 {"entities": [("£1,200", "MONEY")]}),

("Do NHS optical vouchers cover £50 frames or only £25 ones?",
 {"entities": [("NHS", "ORG"), ("£50", "MONEY"), ("£25", "MONEY"), ("optical vouchers", "PRODUCT")]}),

("Are there any benefits for patients receiving treatment between 01 Jan 2026 and 31 Jan 2026?",
 {"entities": [("01 Jan 2026", "DATE"), ("31 Jan 2026", "DATE")]}),

("Does the Learning Support Fund cover placement travel above £30 weekly?",
 {"entities": [("Learning Support Fund", "ORG"), ("£30", "MONEY")]}),

("Am I eligible for an HC2 certificate if my income in 2025–2026 was below £6,000?",
 {"entities": [("HC2 certificate", "PRODUCT"), ("2025–2026", "DATE"), ("£6,000", "MONEY")]}),

("Does student status for the 2026 academic year qualify me for help with health costs?",
 {"entities": [("2026", "DATE")]}),

("Is a part‑time worker earning £900 monthly eligible for the NHS Low Income Scheme?",
 {"entities": [("£900", "MONEY"), ("NHS", "ORG")]}),

("Do pensioners aged over 65 automatically qualify for free prescriptions in 2026?",
 {"entities": [("65", "CARDINAL"), ("2026", "DATE")]}),

]


In [ ]:


# Load base pipeline
nlp = spacy.load("en_core_web_sm")

# Add NER component
ner = nlp.get_pipe("ner")

# Register labels
for text, annot in TRAIN_DATA:
    for entity_text, label in annot["entities"]:
        ner.add_label(label)

# Disable other components for faster training
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]

with nlp.disable_pipes(*other_pipes):
    # Use nlp.initialize() instead of deprecated nlp.begin_training()
    # This method sets up the pipeline components and resources
    nlp.initialize()
    optimizer = nlp.create_optimizer() # Get optimizer from the initialized nlp object

    for epoch in range(30):
        losses = {}
        for text, annot in TRAIN_DATA:
            # Convert string-based entities to offset-based entities
            offset_entities = []
            for entity_text, label in annot["entities"]:
                start = text.find(entity_text)
                if start != -1: # Ensure the entity text is found in the main text
                    end = start + len(entity_text)
                    offset_entities.append((start, end, label))
            # Create a new annotation dictionary with offset-based entities
            processed_annot = {"entities": offset_entities}

            example = Example.from_dict(nlp.make_doc(text), processed_annot)
            nlp.update([example], drop=0.25, sgd=optimizer, losses=losses)
        print(f"Epoch {epoch+1}: Losses {losses}")

Epoch 1: Losses {'ner': np.float32(192.71098)}
Epoch 2: Losses {'ner': np.float32(59.345707)}
Epoch 3: Losses {'ner': np.float32(44.999268)}
Epoch 4: Losses {'ner': np.float32(49.418076)}
Epoch 5: Losses {'ner': np.float32(26.355963)}
Epoch 6: Losses {'ner': np.float32(25.185207)}
Epoch 7: Losses {'ner': np.float32(33.91088)}
Epoch 8: Losses {'ner': np.float32(22.615404)}
Epoch 9: Losses {'ner': np.float32(13.773341)}
Epoch 10: Losses {'ner': np.float32(15.678388)}
Epoch 11: Losses {'ner': np.float32(12.554718)}
Epoch 12: Losses {'ner': np.float32(2.1244915)}
Epoch 13: Losses {'ner': np.float32(0.9524917)}
Epoch 14: Losses {'ner': np.float32(1.7007288)}
Epoch 15: Losses {'ner': np.float32(2.8229847)}
Epoch 16: Losses {'ner': np.float32(0.8286466)}
Epoch 17: Losses {'ner': np.float32(1.9084914)}
Epoch 18: Losses {'ner': np.float32(0.03309498)}
Epoch 19: Losses {'ner': np.float32(0.16216946)}
Epoch 20: Losses {'ner': np.float32(2.0234954)}
Epoch 21: Losses {'ner': np.float32(0.8991727)}


In [ ]:
test_examples = [
    "I paid £42 on 12 Feb 2026 for my dental treatment.",
    "Does NHSBSA accept HC2 certificates for prescriptions?",
    "Is £80 too much for an optical voucher claim?",
    "I travelled on 03 March 2026 for treatment.",
]

for text in test_examples:
    doc = nlp(text)
    print("\nTEXT:", text)
    print("ENTITIES:", [(ent.text, ent.label_) for ent in doc.ents])


TEXT: I paid £42 on 12 Feb 2026 for my dental treatment.
ENTITIES: [('£42', 'MONEY'), ('12 Feb 2026', 'DATE')]

TEXT: Does NHSBSA accept HC2 certificates for prescriptions?
ENTITIES: [('NHSBSA', 'ORG')]

TEXT: Is £80 too much for an optical voucher claim?
ENTITIES: [('£80', 'MONEY'), ('optical voucher', 'PRODUCT')]

TEXT: I travelled on 03 March 2026 for treatment.
ENTITIES: [('03 March 2026', 'DATE')]



#Payslip Information Extraction Using Classification and NER

This short demo shows how Natural Language Processing (NLP) could help reduce manual effort in document‑heavy services  such as IHS (Immigration Health Surcharge), Student Services (Learning Support Fund, Social Work Bursary, Dental Bursary), NHS Help With Health Costs (HC2/HC3 certificates) by automatically extracting key details from a payslip PDF.


It connects the core ideas from our earlier text demo—**cleaning**, **tokenising**, **classifying**, **entity extraction**, and **validation**—to a realistic document scenario.


The goal is not to represent any current live system, but to illustrate how an AI‑assisted workflow might read a payslip, identify sections like *Earnings*, *Deductions*, and *Net Pay*, pull out fields such as dates and amounts, check simple rules (for example, *Net Pay = Earnings − Deductions*), and summarise the results for a reviewer.




# Sample Payslip (Synthetic Example)
<p align="center">
  
  /content/data/sample_payslip.png

</p>


## Overview

This section explains how NLP techniques can be applied to extract information from a payslip.  
The steps shown here directly relate to the NLP concepts demonstrated earlier:

- Text cleaning  
- Tokenisation  
- Classification  
- Entity extraction  
- Validation  

These steps turn an unstructured payslip PDF or image into structured, meaningful data.


## Step 1 — OCR: Convert PDF/Image to Text

A payslip is normally submitted as a PDF or a scanned image.  
Before NLP can interpret it, text must be extracted using OCR tools such as:

- Amazon Textract  
- Google Document AI  
- Azure Form Recognizer  

OCR produces:
- Lines of text  
- Words  
- Detected tables  
- Key-value pairs  

This serves as the "raw text" input for further NLP processing.

## Step 2 — Text Cleaning

OCR output often contains noise such as:
- Extra spaces  
- Line breaks  
- Strange characters  
- Misread letters or digits  

Cleaning tasks include:
- Converting to lowercase  
- Removing punctuation  
- Normalising dates and currency values  
- Removing unnecessary symbols  

This mirrors the text‑cleaning step used in the earlier NLP demo.

## Step 3 — Tokenisation

After cleaning, the text is split into smaller pieces called tokens.

Examples:
- "basic", "pay"
- "37.5", "hrs"
- "net", "pay"
- "2150.00"
- "01", "feb", "2026"

Same process as earlier:  
Convert text into meaningful building blocks for analysis.

## Step 4 —  Classification

Just like "intent classification" earlier, we classify segments of the payslip into sections:

- Employer information  
- Employee information  
- Pay period  
- Earnings  
- Deductions  
- Net pay  

This gives structure to the extracted text and helps identify where important information lies.

## Step 5 — Named Entity Extraction

From each classified section, NLP extracts key pieces of information.

### Examples of extracted entities:
- Employer name  
- Employee name  
- NI number  
- Tax code  
- Pay period start and end dates  
- Hours worked  
- Gross pay, deductions, and net pay  

This is similar to identifying keywords or entities from a sentence in earlier examples.

## Step 6 — Structured Output Summary

Once NLP extraction and validation are complete, the information is presented in structured form.

Example output:

- Employer: Northshire NHS Trust  
- Employee: A. Brown  
- Pay Period: 01 Feb 2026 – 28 Feb 2026  
- Gross Pay: £2,307.50  
- Deductions: £539.57  
- Net Pay: £1,767.93  

This makes the payslip easy for a system or a human reviewer to understand.



The same NLP concepts demonstrated earlier—cleaning, tokenising, classifying and entity extraction can be applied to real documents such as payslips.

This turns a PDF or image into structured, usable information, showing how NLP underpins many practical document‑processing workflows.
